# 06 - Prompt Agent on the Connected Model

Run [02_configure.ipynb](02_configure.ipynb) first. Use **Python (byom01-capability-testing)** with the working directory set to `notebooks/`.

This notebook creates a prompt agent using the existing connected model. The agent has no tools, including web search. Its system instructions tell it to speak like a pirate, and the user asks it to explain Azure AI Foundry.

**Paid inference:** the invocation makes one Foundry Responses request. Run only with approval.

The cleanup section deletes only this notebook's test conversation and agent version. It leaves the connected model and gateway configuration unchanged.


## Load Gateway Settings


In [ ]:
import json
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential

demo_root = Path("..").resolve()
gateway = json.loads((demo_root / "env" / "gateway.json").read_text(encoding="utf-8"))
credential = AzureCliCredential(subscription=gateway["subscriptionId"])
project_client = AIProjectClient(endpoint=gateway["projectEndpoint"], credential=credential)
openai_client = project_client.get_openai_client(max_retries=0)
print(f"Project: {gateway['projectId']}")
print(f"Connected model: {gateway['modelReference']}")


## Create Prompt Agent

Create a prompt agent on the registered connected model. No tool definitions are supplied, so the agent cannot use web search.


In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

agent = project_client.agents.create_version(
    agent_name="byom01-pirate-agent",
    definition=PromptAgentDefinition(
        model=gateway["modelReference"],
        instructions=(
            "You are a helpful assistant. Always speak like a pirate. "
            "Do not use tools or web search."
        ),
    ),
)
print(f"Agent: {agent.name}, version: {agent.version}")


## Invoke Prompt Agent

Ask the agent to explain Azure AI Foundry. The response should contain pirate-style wording while answering the question.


In [ ]:
conversation = openai_client.conversations.create()
response = openai_client.responses.create(
    conversation=conversation.id,
    input="Explain what Azure AI Foundry is.",
    extra_body={"agent_reference": {
        "name": agent.name, "type": "agent_reference",
    }},
)
assert response.status == "completed", f"Foundry response did not complete: {response.status}"
assert response.output_text.strip(), "Foundry returned no output text"
print(response.output_text)
print(f"Foundry response: {response.id}")


## Clean Up Test Resources

Run this when finished inspecting the response. It deletes only the conversation and agent version created by this notebook; it does not delete the connected model or gateway.


In [ ]:
openai_client.conversations.delete(conversation_id=conversation.id)
project_client.agents.delete_version(
    agent_name=agent.name, agent_version=agent.version, force=True,
)
print("Test conversation and agent version deleted; connection and gateway retained.")
